# Embed and Upsert FIVB Rules to Qdrant

Load Docling chunks, create Ollama dense embeddings, create BM25 sparse vectors, upsert both into Qdrant, and run a hybrid retrieval smoke test.

In [ ]:
from pathlib import Path
import json
import os
import uuid

from dotenv import load_dotenv
from fastembed import SparseTextEmbedding
import ollama
from qdrant_client import QdrantClient, models

load_dotenv()

COLLECTION_NAME = "volleyball_rules_hybrid"
CHUNKS_PATH = Path("cleaned_fivb_rules_chunks.jsonl")

DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "sparse"
DENSE_MODEL = "qwen3-embedding:0.6b"
DENSE_VECTOR_SIZE = 1024
SPARSE_MODEL = "Qdrant/bm25"
BATCH_SIZE = 16

QDRANT_ENDPOINT = os.environ["QDRANT_ENDPOINT"]
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]

In [ ]:
assert CHUNKS_PATH.exists(), f"Missing {CHUNKS_PATH}. Run test.ipynb first to generate chunks."

with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

assert records, "No chunk records found"

print(f"Loaded {len(records)} chunk records from {CHUNKS_PATH}")
print(records[0]["id"])
print(records[0]["text"][:500])

In [ ]:
client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config={
            DENSE_VECTOR_NAME: models.VectorParams(
                size=DENSE_VECTOR_SIZE,
                distance=models.Distance.COSINE,
            )
        },
        sparse_vectors_config={
            SPARSE_VECTOR_NAME: models.SparseVectorParams(),
        },
    )

collection = client.get_collection(COLLECTION_NAME)
print(f"Collection: {COLLECTION_NAME}")
print(f"Status: {collection.status}")
print(f"Points: {collection.points_count}")

In [ ]:
sparse_model = SparseTextEmbedding(SPARSE_MODEL)

def point_id(record):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, record["id"]))

def to_sparse_vector(sparse_embedding):
    return models.SparseVector(
        indices=sparse_embedding.indices.tolist(),
        values=sparse_embedding.values.tolist(),
    )

def payload_from_record(record):
    metadata = record.get("metadata", {})
    return {
        "chunk_id": record["id"],
        "source": record["source"],
        "chunk_index": record["chunk_index"],
        "text": record["text"],
        "raw_text": record["raw_text"],
        "headings": metadata.get("headings"),
        "captions": metadata.get("captions"),
    }

def batches(items, batch_size):
    for start in range(0, len(items), batch_size):
        yield start, items[start:start + batch_size]

In [ ]:
# Dry run: embed one chunk and build one Qdrant point before upserting everything.
sample = records[0]
sample_text = sample["text"]

sample_dense = ollama.embed(
    model=DENSE_MODEL,
    input=sample_text,
    truncate=False,
)["embeddings"][0]
sample_sparse = next(sparse_model.passage_embed([sample_text]))

assert len(sample_dense) == DENSE_VECTOR_SIZE, len(sample_dense)

sample_point = models.PointStruct(
    id=point_id(sample),
    vector={
        DENSE_VECTOR_NAME: sample_dense,
        SPARSE_VECTOR_NAME: to_sparse_vector(sample_sparse),
    },
    payload=payload_from_record(sample),
)

print(f"Dense vector size: {len(sample_dense)}")
print(f"Sparse non-zero terms: {len(sample_sparse.indices)}")
print(f"Point id: {sample_point.id}")
print(sample_point.payload["text"][:500])

In [ ]:
upserted = 0

for start, batch in batches(records, BATCH_SIZE):
    texts = [record["text"] for record in batch]

    dense_vectors = ollama.embed(
        model=DENSE_MODEL,
        input=texts,
        truncate=False,
    )["embeddings"]

    sparse_vectors = list(sparse_model.passage_embed(texts))

    points = []
    for record, dense_vector, sparse_vector in zip(batch, dense_vectors, sparse_vectors):
        assert len(dense_vector) == DENSE_VECTOR_SIZE, len(dense_vector)
        points.append(
            models.PointStruct(
                id=point_id(record),
                vector={
                    DENSE_VECTOR_NAME: dense_vector,
                    SPARSE_VECTOR_NAME: to_sparse_vector(sparse_vector),
                },
                payload=payload_from_record(record),
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
        wait=True,
    )

    upserted += len(points)
    print(f"Upserted {upserted}/{len(records)}")

In [ ]:
collection = client.get_collection(COLLECTION_NAME)
print(f"Collection points: {collection.points_count}")

In [ ]:
def hybrid_search(query, limit=5, prefetch_limit=20):
    dense_query = ollama.embed(
        model=DENSE_MODEL,
        input=query,
        truncate=False,
    )["embeddings"][0]
    sparse_query = to_sparse_vector(next(sparse_model.query_embed(query)))

    return client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            models.Prefetch(
                query=dense_query,
                using=DENSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
            models.Prefetch(
                query=sparse_query,
                using=SPARSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
        ],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=limit,
        with_payload=True,
    )

results = hybrid_search("How many contacts does a team have to return the ball?")

for hit in results.points:
    payload = hit.payload or {}
    print(f"score={hit.score:.4f} chunk={payload.get('chunk_id')}")
    print(f"headings={payload.get('headings')}")
    print(payload.get("text", "")[:500])
    print("-" * 80)